In [1]:
import os
os.environ["TORCH_LOGS"] = "+output_code"
os.environ["TORCH_LOGS_OUT"] = "compiled_kernels.log"

In [ ]:
import torch
import torch.nn as nn

from optimizer.ademamix import AdEMAMix as AdemamixForloop
from efficient_optimizer.ademamix import AdEMAMix as AdemamixForeach
import torch._inductor.metrics
import torch._dynamo as dynamo
dynamo.config.recompile_limit = 8

HIDDEN_DIM = 16
NUM_LAYERS = 3
NUM_STEPS = 100

In [3]:
import torch.nn as nn

def _build_model(device: torch.device, dtype: torch.dtype) -> nn.Module:
    torch.manual_seed(0)
    layers = [nn.Linear(HIDDEN_DIM, HIDDEN_DIM, bias=True) for _ in range(NUM_LAYERS)]
    return nn.Sequential(*layers).to(device=device, dtype=dtype)

def _apply_random_grads(model_a: nn.Module) -> None:
    torch.manual_seed(0)
    a = dict(model_a.named_parameters())
    for name in a.keys():
        g = torch.randn_like(a[name].data)
        a[name].grad = g


In [ ]:
device = 'cuda'
dtype = torch.float16
torch._inductor.metrics.generated_kernel_count = 0
model_baseline = _build_model(device, dtype)

opt_baseline = torch.optim.AdamW(model_baseline.parameters(), lr=1e-2)


opt_baseline.step = torch.compile(opt_baseline.step)

for step in range(1, NUM_STEPS + 1):
    _apply_random_grads(model_baseline)
    opt_baseline.step()
print(f"Generated Triton Kernels: {torch._inductor.metrics.generated_kernel_count}")


V0309 15:50:20.262000 976540 torch/_inductor/graph.py:2345] [1/0] [__output_code] Output code: 
V0309 15:50:20.262000 976540 torch/_inductor/graph.py:2345] [1/0] [__output_code] # AOT ID: ['3_inference']
V0309 15:50:20.262000 976540 torch/_inductor/graph.py:2345] [1/0] [__output_code] from ctypes import c_void_p, c_long, c_int
V0309 15:50:20.262000 976540 torch/_inductor/graph.py:2345] [1/0] [__output_code] import torch
V0309 15:50:20.262000 976540 torch/_inductor/graph.py:2345] [1/0] [__output_code] import math
V0309 15:50:20.262000 976540 torch/_inductor/graph.py:2345] [1/0] [__output_code] import random
V0309 15:50:20.262000 976540 torch/_inductor/graph.py:2345] [1/0] [__output_code] import os
V0309 15:50:20.262000 976540 torch/_inductor/graph.py:2345] [1/0] [__output_code] import tempfile
V0309 15:50:20.262000 976540 torch/_inductor/graph.py:2345] [1/0] [__output_code] from math import inf, nan
V0309 15:50:20.262000 976540 torch/_inductor/graph.py:2345] [1/0] [__output_code] from c

Generated Triton Kernels: 2


In [ ]:
torch._inductor.metrics.generated_kernel_count = 0
from efficient_optimizer.ademamix import ademamix_foreach_fn
device = 'cuda'
dtype = torch.float32

model_efficient = _build_model(device, dtype)

opt_efficient = AdemamixForeach(model_efficient.parameters(), lr=1e-2, weight_decay=0.1, alpha_warmup=51, beta3_warmup=51)


for step in range(1, NUM_STEPS + 1):
    _apply_random_grads(model_efficient)
    opt_efficient.step()
print(f"Generated Triton Kernels: {torch._inductor.metrics.generated_kernel_count}")


V0309 16:02:20.961000 1001617 torch/_inductor/graph.py:2345] [0/0] [__output_code] Output code: 
V0309 16:02:20.961000 1001617 torch/_inductor/graph.py:2345] [0/0] [__output_code] # AOT ID: ['0_inference']
V0309 16:02:20.961000 1001617 torch/_inductor/graph.py:2345] [0/0] [__output_code] from ctypes import c_void_p, c_long, c_int
V0309 16:02:20.961000 1001617 torch/_inductor/graph.py:2345] [0/0] [__output_code] import torch
V0309 16:02:20.961000 1001617 torch/_inductor/graph.py:2345] [0/0] [__output_code] import math
V0309 16:02:20.961000 1001617 torch/_inductor/graph.py:2345] [0/0] [__output_code] import random
V0309 16:02:20.961000 1001617 torch/_inductor/graph.py:2345] [0/0] [__output_code] import os
V0309 16:02:20.961000 1001617 torch/_inductor/graph.py:2345] [0/0] [__output_code] import tempfile
V0309 16:02:20.961000 1001617 torch/_inductor/graph.py:2345] [0/0] [__output_code] from math import inf, nan
V0309 16:02:20.961000 1001617 torch/_inductor/graph.py:2345] [0/0] [__output_co

Generated Triton Kernels: 1
